# Task 2B - 06 Validation and comparison
Every candidate allocation is checked two independent ways - the **unmodified official validator** and an **independent re-implementation of all rules** - and compared with the first solver's submitted file.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../../tools/task2b')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time
from task2b_common import *
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 120)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .25})
for d in (METRICS, FIGURES, INTERIM): d.mkdir(parents=True, exist_ok=True)
scn, fleet, veh, dt, al = load_tables()
sub_team = pd.read_csv(SUBMISSION)            # the teammate's submitted allocation (read-only)
import task2b_exact as X
S = X.S


In [2]:
cands = {'first solver (submitted)': sub_team}
for pol in ('weighted', 'lexicographic'):
    cands['exact: ' + pol] = X.to_submission(X.solve_exact(pol))
tmp = INTERIM / 'candidates'; tmp.mkdir(exist_ok=True)
rows, checks = [], {}
for name, s in cands.items():
    p = tmp / (name.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '') + '.csv'); s.to_csv(p, index=False)
    ok, msg = official_check(p)
    ind = independent_check(pd.read_csv(p), scn, fleet, veh, dt, al)
    checks[name] = ind.set_index('check').violations
    rows.append((name, ok, int(ind.violations.sum())))
pd.DataFrame(rows, columns=['candidate', 'official validator passed', 'independent-check violations'])

,candidate,official validator passed,independent-check violations
0,first solver (submitted),True,0
1,exact: weighted,True,0
2,exact: lexicographic,True,0


In [3]:
pd.DataFrame(checks)

,first solver (submitted),exact: weighted,exact: lexicographic
check,,,
every order decided exactly once,0,0,0
deferred rows have blank vehicle/trip,0,0,0
only vehicles marked available,0,0,0
R1 one brand and one district per trip,0,0,0
R2 chilled only on reefer,0,0,0
R3 van_only outlets only on vans,0,0,0
R4 vehicle serves only its home depot,0,0,0
R6 volume within capacity,0,0,0
R6 weight within capacity,0,0,0


All three allocations are feasible under both checks. The independent re-implementation shares no code with the solver or with the official script, so agreement is meaningful.

## Head-to-head

In [4]:
def summary(name, s):
    m = s[s.decision == 'served'].merge(scn, on=['scenario', 'order_ref', 'outlet_id'])
    st, late = simulate_submission(s, S)
    t = trip_table(s, scn, veh, dt, al)
    return dict(served=len(m), deferred=int((s.decision == 'deferred').sum()), m3_served=round(m.order_volume_m3.sum(), 1), chilled_served=int((m.temp_requirement == 'chilled').sum()),
                vehicles=m.vehicle_id.nunique(), trips=len(t), late_stops=late, max_trip_min=int(t.trip_min.max()), longest_trip_stops=int(t.stops.max()),
                deferred_yesterday_served=int(m.deferred_yesterday.sum()))
cmp_ = pd.DataFrame({n: summary(n, s) for n, s in cands.items()}); cmp_.to_csv(METRICS / 'exact_comparison.csv'); cmp_

,first solver (submitted),exact: weighted,exact: lexicographic
served,79.0,79.0,77.0
deferred,6.0,6.0,8.0
m3_served,328.3,328.3,313.8
chilled_served,21.0,21.0,19.0
vehicles,13.0,15.0,15.0
trips,23.0,25.0,25.0
late_stops,8.0,3.0,1.0
max_trip_min,270.0,266.0,266.0
longest_trip_stops,10.0,8.0,8.0
deferred_yesterday_served,9.0,9.0,10.0


**Reading.**
- *Same service outcome, better execution:* the exact weighted plan serves exactly what the first solver served, but with no avoidable late stop and no 10-stop 4-hour ambient trips.
- *The late stops that remain* in the exact plans (the cell above asserts that every one of them is on a reefer's **second Fresh trip**) arise because a literal clock starts that trip only after the first trip's return leg. The booklet states its budgets already allow for the return, and its feasibility rules do not include windows, so these are not defects of the plan - but they are the one place where a literal clock disagrees with the booklet's abstraction, and we report them rather than hide them.

## Which stops are late, and why

In [5]:
for name in ('first solver (submitted)', 'exact: weighted', 'exact: lexicographic'):
    st, late = simulate_submission(cands[name], S)
    t = trip_table(cands[name], scn, veh, dt, al)[['vehicle_id', 'trip_id', 'vehicle', 'district', 'stops', 'trip_min']]
    x = st[st.late == 1].merge(t, on=['vehicle_id', 'trip_id'])[['order_ref', 'vehicle_id', 'trip_id', 'vehicle', 'district', 'stops', 'depart_depot', 'arrive', 'window']]
    other = x[~((x.trip_id == 2) & x.vehicle.str.contains('reefer'))]
    print('%-26s late stops: %d | of which NOT on a reefer second trip: %d' % (name, late, len(other)))
    if name != 'first solver (submitted)':
        assert len(other) == 0, 'an exact plan has a late stop outside reefer second trips'
    print(x.to_string(index=False)); print()

first solver (submitted)   late stops: 8 | of which NOT on a reefer second trip: 4
order_ref vehicle_id  trip_id       vehicle district  stops depart_depot arrive      window
   S1-012     VEH003        2  truck/reefer  Colombo      1        09:07  09:31 05:30-08:00
   S1-046     VEH007        2  truck/reefer Kalutara      3        06:40  07:44 05:00-07:30
   S1-051     VEH007        2  truck/reefer Kalutara      3        06:40  08:12 05:00-07:30
   S1-048     VEH007        2  truck/reefer Kalutara      3        06:40  08:39 03:00-08:00
   S1-032     VEH024        1 truck/ambient  Gampaha     10        03:30  08:14 05:30-08:00
   S1-039     VEH024        1 truck/ambient  Gampaha     10        03:30  08:38 05:30-08:00
   S1-019     VEH030        1 truck/ambient  Colombo     10        03:30  08:06 05:30-08:00
   S1-022     VEH030        1 truck/ambient  Colombo     10        03:30  08:29 05:30-08:00

exact: weighted            late stops: 3 | of which NOT on a reefer second trip: 0
order

## Deferral analysis: unavoidable vs chosen

In [6]:
rows = []
for name, s in cands.items():
    for r in s[s.decision == 'deferred'].order_ref:
        o = S.ORDERS[r]; why = S.unavoidable_reason(o, S.AVAILABLE['S1'])
        rows.append((name, r, o.outlet, o.brand, o.district, 'chilled' if o.chilled else 'ambient', round(o.m3, 1), o.deferred_yday, o.days_since, 'unavoidable' if why else 'capacity (reefer)'))
dd = pd.DataFrame(rows, columns=['candidate', 'order_ref', 'outlet', 'brand', 'district', 'type', 'm3', 'deferred_yesterday', 'days_since', 'kind'])
dd.to_csv(METRICS / 'exact_deferral_comparison.csv', index=False); dd.sort_values(['candidate', 'order_ref'])

,candidate,order_ref,outlet,brand,district,type,m3,deferred_yesterday,days_since,kind
12,exact: lexicographic,S1-003,OUT002,Fresh,Colombo,chilled,1.8,0,1,capacity (reefer)
13,exact: lexicographic,S1-058,OUT054,Fresh,Galle,chilled,16.5,0,1,capacity (reefer)
14,exact: lexicographic,S1-064,OUT060,Fresh,Matara,chilled,6.8,0,1,capacity (reefer)
15,exact: lexicographic,S1-067,OUT062,Fresh,Matara,chilled,5.2,0,1,capacity (reefer)
16,exact: lexicographic,S1-071,OUT065,Fresh,Kurunegala,chilled,12.0,0,1,capacity (reefer)
17,exact: lexicographic,S1-073,OUT066,Fresh,Kurunegala,chilled,5.8,0,1,capacity (reefer)
18,exact: lexicographic,S1-075,OUT067,Fresh,Kurunegala,chilled,7.2,0,2,capacity (reefer)
19,exact: lexicographic,S1-078,OUT070,Style,Kurunegala,ambient,40.7,0,2,unavoidable
6,exact: weighted,S1-056,OUT053,Fresh,Galle,chilled,3.8,0,2,capacity (reefer)
7,exact: weighted,S1-058,OUT054,Fresh,Galle,chilled,16.5,0,1,capacity (reefer)


**Finding.** One deferral (S1-078) is physically unavoidable under every policy. All other deferrals are reefer-capacity deferrals, and *which* chilled orders wait is the policy choice.